## **INFO-61048 (26W) Natural Language Processing:**
# **Project: Ops Knowledge Base AI**

**Author:** Yun-Jiung Wang

**Student Number:** 1256222

**Date:** March 10th, 2026

## 1. Executive Summary
**FalconOps** is an advanced IT support tool designed to transform raw, unstructured expert communication logs into a searchable knowledge base. By leveraging **Natural Language Processing (NLP)**, the system assists system administrators in retrieving technical solutions from over 8,000 historical expert records. Unlike traditional keyword-based searches, FalconOps understands the context and intent behind user queries, providing a more reliable and efficient troubleshooting experience.

---

## 2. Problem Statement
Technical support data, particularly from IRC logs or internal chats, is often fragmented and informal. This leads to two major challenges:
* **Information Overload:** It is difficult for junior engineers to find specific solutions within thousands of lines of text.
* **Data Noise:** Informal language and "slang" can make expert advice difficult to interpret or apply in a professional environment.

---

## 3. Technical Architecture
The system is built upon a **Decoupled Architecture**, separating data processing from the user interface to ensure modularity and scalability.



### A. Data Processing Pipeline (NLP_Project)
* **Sentence Embeddings:** Utilizing the `all-MiniLM-L6-v2` Transformer model to convert text instructions into high-dimensional vectors.
* **Vector Storage:** Efficient storage of 8,000+ embeddings using `.npy` format for rapid mathematical comparison.

### B. Logic & Reasoning Engine (brain.py)
* **Semantic Matching:** Uses **Cosine Similarity** to identify the closest expert response to a user's query.
* **Confidence Thresholding:** A critical safety feature that blocks responses with a confidence score below **0.75**, preventing the system from providing irrelevant or incorrect advice.
* **Post-Processing:** A refinement layer that filters informal language and formats the output into a professional advisory tone.

### C. Interface Layer (app.py)
* A streamlined dashboard built with **Streamlit**, providing real-time feedback and a clear escalation path to human experts (Fanshawe Tech Support) when the AI confidence is low.

---

## 4. Key Features
* **Intent Recognition:** Understands technical synonyms (e.g., "modify permissions" vs. "chown").
* **Fail-Safe Mechanism:** Automatically triggers an escalation protocol if no high-confidence solution is found.
* **Optimized Performance:** Uses pre-calculated vector caches to ensure sub-second response times.

---

## 5. Conclusion & Future Work
FalconOps demonstrates the effective application of **Retrieval-Augmented Logic** in a DevOps environment. While the current version focuses on semantic retrieval, future iterations will explore **Style Transfer** models to automatically rewrite informal logs into standardized technical documentation.



## Enviromnent Setup

In [1]:
# NLP & Vector Search
try:
    import datasets, sentence_transformers, faiss
except ImportError:
    !pip install -q datasets sentence-transformers faiss-cpu

# # System-level dependencies (Poppler for PDF and Tesseract for OCR)
# print("Checking system tools...")
# !sudo apt-get update -qq
# !sudo apt-get install -y -qq poppler-utils tesseract-ocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 72.8 MB/s eta 0:00:00


## Import Libs

In [17]:
from datasets import load_dataset

import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
import faiss

# =========================
# Visualization
# =========================
import matplotlib.pyplot as plt
import seaborn as sns

import ast
import pickle
import os

# =========================
# Cluster
# =========================
from sklearn.cluster import KMeans
import pandas as pd
import joblib

import warnings
warnings.filterwarnings("ignore")

## Load the dataset

In [4]:
# Load dataset
dataset = load_dataset("sedthh/ubuntu_dialogue_qa",verification_mode="no_checks")
# train dataset
df = pd.DataFrame(dataset['train'])
display(df.head())

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-bb5f874d67d84f(…):   0%|          | 0.00/2.16M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16181 [00:00<?, ? examples/s]

,INSTRUCTION,RESPONSE,SOURCE,METADATA
0,"hi, is there a CLI command to roll back any up...",your recourse is to re-install fresh the older...,ubuntu-dialogue,"{""user_question"": ""edd"", ""user_answer"": ""n8tus..."
1,A LiveCD iso can be burned to a DVD-R and run ...,"I hope so, or the custom DVDs I've done are wo...",ubuntu-dialogue,"{""user_question"": ""usrl"", ""user_answer"": ""Ghos..."
2,"hello, is there a way to adjust gamma settings...",for me i have my nvidia settings manager and i...,ubuntu-dialogue,"{""user_question"": ""nucco_"", ""user_answer"": ""sp..."
3,does ubuntu come with a firewall by default?,no iptables rule is loaded by deault on ubuntu,ubuntu-dialogue,"{""user_question"": ""aeleon"", ""user_answer"": ""er..."
4,Can someone tell me howto get rid of Google Ch...,sudo dpkg -l |grep -i chrom ----> sudo apt-get...,ubuntu-dialogue,"{""user_question"": ""frold"", ""user_answer"": ""shi..."


## Pre-Processing

In this section, we remove the null response from the dataset, and split the dataset into train and test. Then to remove the noise, we get the Questioner and Anserer username data from the METADATA column.

In [5]:
# Remove null
df = df.dropna()

# Split train/test
train_dataset = df.sample(frac=0.8, random_state=42)
test_dataset = df.drop(train_dataset.index)

# Get Users
def extract_answerer(metadata_str):
    try:
        # JSON string to dic
        meta_dict = ast.literal_eval(metadata_str) if isinstance(metadata_str, str) else metadata_str
        q_user = meta_dict.get('user_question', 'unq')
        a_user = meta_dict.get('user_answer', 'una')
        return pd.Series([q_user, a_user])
    except:
        return pd.Series(["unq","una"])


df[['QUESTIONER', 'SOLVER']] = df['METADATA'].apply(extract_answerer)
df['SOLVER'] = df['SOLVER'].str.lower()
df['QUESTIONER'] = df['QUESTIONER'].str.lower()

df = df.drop(columns=['METADATA','SOURCE'])
display(df.tail())
print(len(df))

,INSTRUCTION,RESPONSE,QUESTIONER,SOLVER
16168,is there any GUI irc client besides pidgin ?,xchat,jameela,plong0
16169,"Hello , if I have a log file and i like to see...",you can try watch 'tail /path/to/logfile',zedde,adam7
16170,guys im trying to install itask but when i try...,sudo aptitude install automake autoconf build-...,silvernode,estreyela
16171,is there anyway to recurse with sftp in it's n...,"I believe not, but try lftp instead (it suppor...",psion,seveas
16172,how do i set permissions on /dir so that new s...,-> erUSUL is correct i forgot about those setu...,drx,n8tuser


16173


### Observe top 50 SOLVERS to define Golden medals responser

To get the higer quality datasets, we will only remain the top 35 solvers response (since the min number of top 50 responser is 35).



In [6]:
# top 50 response
solver_counts =  df['SOLVER'].value_counts().head(50)
print(f"min of solver counts: {min(solver_counts)}")

solver_cnt = df['SOLVER'].value_counts()

print(f"mean:{solver_counts.mean()}")
print(f"median:{solver_counts.median()}")

min of solver counts: 35
mean:79.26
median:52.5


### Remain toppers' response

Here we will only remain those toppers' response to make sure the response quality. Some only reply once, so their response might not be accurate.

In [7]:
# Get solvers
top_solvers = solver_counts[solver_counts>=35].index

# Remain toppers' response
df_pro = df[df['SOLVER'].isin(top_solvers)].copy()

## Remove short response
df_refined = df_pro[df_pro['RESPONSE'].str.len() > 30].copy()

display(df_refined.head())

,INSTRUCTION,RESPONSE,QUESTIONER,SOLVER
3,does ubuntu come with a firewall by default?,no iptables rule is loaded by deault on ubuntu,aeleon,erusul
7,is there a way to see if a hard disk has bad b...,"have you considered, however, monitoring your ...",arthurmaciel,ljl
11,"Hi, My Western Digital USB Passport Drive does...",I have one of those too.. it works nicely in u...,steve176,slart
17,Where can I find full DVD of hardy for amd64 ???,http://www.acc.umu.se/~mighty/ubuntu/ubuntu-8....,mad_max02,maco
19,hi folks. i am trying to convert screencast I ...,you probably the ffmpeg binaries from medibunt...,jmut,erusul


### Get Questions and Anwers list for embending and save it into files or UI

In [8]:
questions = df_refined["INSTRUCTION"].tolist()
answers = df_refined["RESPONSE"].tolist()

print(f"question num:{len(questions)}; answers num:{len(answers)}")

question num:2534; answers num:2534


## Create Model

We have selected SentenceTransformer as our core model due to its superior ability to handle semantic retrieval within the Ubuntu dataset. This model utilizes dense vector embeddings to capture the true intent of user queries. By mapping questions into a high-dimensional vector space, the system performs efficient similarity searches to identify the most relevant archived solutions. This approach ensures high precision in technical troubleshooting while maintaining low computational overhead. Ultimately, SentenceTransformer provides a reliable, scalable, and fact-based foundation, delivering verified community knowledge to users with near-instantaneous response times.

In [9]:
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Embedding the questions:

To enable efficient semantic retrieval, we performed Text-to-Vector Embedding on the expert knowledge base. By converting natural language into high-dimensional vectors, the system can measure semantic similarity rather than just keyword matching. These vectors are persisted as .npy files to ensure low-latency inference during real-time user interaction.

In [10]:
question_embeddings = model.encode(questions, show_progress_bar=True)
print(question_embeddings)

Batches:   0%|          | 0/80 [00:00<?, ?it/s]

[[ 0.07549541  0.01222624 -0.0249166  ...  0.02297675  0.04964383
   0.01901241]
 [-0.02366473 -0.02739873 -0.04572623 ... -0.05522996 -0.00368845
   0.06834889]
 [-0.00618826 -0.0267509   0.03013163 ...  0.01518088 -0.03224819
   0.06129985]
 ...
 [ 0.00444519  0.00379755 -0.07719884 ...  0.02253566  0.00730566
  -0.09924315]
 [ 0.01187467  0.01458819 -0.01999028 ...  0.02509865 -0.01897434
   0.03461244]
 [ 0.02191286 -0.03294545 -0.02390194 ...  0.06107355  0.06099329
   0.0477547 ]]


## Clustering the Questions

For Frequently asked questions and answers, and catogortize the user query questions

In [15]:
num_clusters = 10
kmeans = KMeans(n_clusters=num_clusters, n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(question_embeddings) # use current question_embedding

# Save to
df_final = pd.DataFrame({
    'cluster_id': cluster_labels,
    'question': questions,
    'answer': answers
})

display(df_final.head())
print(len(df_final))

,cluster_id,question,answer
0,1,does ubuntu come with a firewall by default?,no iptables rule is loaded by deault on ubuntu
1,7,is there a way to see if a hard disk has bad b...,"have you considered, however, monitoring your ..."
2,7,"Hi, My Western Digital USB Passport Drive does...",I have one of those too.. it works nicely in u...
3,8,Where can I find full DVD of hardy for amd64 ???,http://www.acc.umu.se/~mighty/ubuntu/ubuntu-8....
4,8,hi folks. i am trying to convert screencast I ...,you probably the ffmpeg binaries from medibunt...


2534


## Data Serialization for Deployment

We utilized NumPy binary format (.npy) and Pickle (.pkl) to serialize our processed knowledge base. This allows the Streamlit UI to load the pre-computed embeddings and response strings instantly without re-processing the entire dataset (8,000+ records) every time the application starts. This ensures low-latency and high performance for the end-user experience.

In [16]:
print(f"File would be saved in: {os.getcwd()}")

# save vector
np.save('expert_vectors.npy', question_embeddings)
# save df_final answer
with open('expert_answers.pkl', 'wb') as f:
    # pickle.dump(df_final, f)
    # df_final.to_pickle(f)
    pickle.dump(df_final, f)

print("Saved expert answers to 'expert_answers.pkl'")

File would be saved in: /content
Saved expert answers to 'expert_answers.pkl'


## Showing the User Interface

#### Set up the environment

In [ ]:
!pip install -q streamlit sentence_transformers scikit-learn

!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

--2026-04-09 20:29:09--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb [following]
--2026-04-09 20:29:09--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/ec689fe1-d727-4ebd-bbc3-5967730ab54e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-09T21%3A09%3A07Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

### Execue the UI file

In [ ]:
# 1. kill the old process
!pkill -f streamlit
!pkill -f cloudflared

# 2. Use nohup to ensure Streamlit runs stably in the background
!nohup streamlit run app.py --server.port 8501 &

# 3. Wait 15 seconds for the brain to load the model
import time
print("🧠 Loading expert knowledge base, please wait 15 seconds...")
time.sleep(15)

# 4. restart cloudflared tunnel
print("Building encrypted tunnel...")
!cloudflared tunnel --url http://localhost:8501

nohup: appending output to 'nohup.out'
🧠 Loading expert knowledge base, please wait 15 seconds...
Building encrypted tunnel...
2026-04-09T21:20:31Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-09T21:20:31Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-09T21:20:34Z INF +--------------------------------------------------------------------------------------------+
2026-04-09T21:20:34Z INF |  Your quick Tunnel has been created! Visit it a